In [1]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.dummy import DummyClassifier
import mlflow
import mlflow.sklearn


In [2]:
path = '../data/processed/telco_customer_churn_processed.csv'

In [3]:
df = pd.read_csv(path)

In [4]:
df_engineered = df.copy()

In [5]:
X = df_engineered.drop(columns=['target'])
X.columns.tolist()

['Tenure Months',
 'Churn Score',
 'CLTV',
 'Gender_Male',
 'Partner_Yes',
 'Dependents_Yes',
 'Phone Service_Yes',
 'Multiple Lines_No phone service',
 'Multiple Lines_Yes',
 'Internet Service_Fiber optic',
 'Internet Service_No',
 'Online Security_No internet service',
 'Online Security_Yes',
 'Online Backup_No internet service',
 'Online Backup_Yes',
 'Device Protection_No internet service',
 'Device Protection_Yes',
 'Tech Support_No internet service',
 'Tech Support_Yes',
 'Streaming TV_No internet service',
 'Streaming TV_Yes',
 'Streaming Movies_No internet service',
 'Streaming Movies_Yes',
 'Contract_One year',
 'Contract_Two year',
 'Paperless Billing_Yes',
 'Payment Method_Credit card (automatic)',
 'Payment Method_Electronic check',
 'Payment Method_Mailed check']

In [6]:
y = df_engineered['target']
X = df_engineered.drop(columns=['target'])
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

selector_preview = SelectKBest(f_classif, k=min(25, X_train.shape[1]))
X_train_selected = selector_preview.fit_transform(X_train, y_train)

X_test_selected = selector_preview.transform(X_test)

selected_mask = selector_preview.get_support()
selected_features = [name for name, keep in zip(feature_names, selected_mask) if keep]
print(f"\nFeatures selecionadas ({len(selected_features)}):")
for i, feat in enumerate(selected_features, 1):
    print(f"{i}. {feat}")


Features selecionadas (25):
1. Tenure Months
2. Churn Score
3. CLTV
4. Partner_Yes
5. Dependents_Yes
6. Internet Service_Fiber optic
7. Internet Service_No
8. Online Security_No internet service
9. Online Security_Yes
10. Online Backup_No internet service
11. Online Backup_Yes
12. Device Protection_No internet service
13. Device Protection_Yes
14. Tech Support_No internet service
15. Tech Support_Yes
16. Streaming TV_No internet service
17. Streaming TV_Yes
18. Streaming Movies_No internet service
19. Streaming Movies_Yes
20. Contract_One year
21. Contract_Two year
22. Paperless Billing_Yes
23. Payment Method_Credit card (automatic)
24. Payment Method_Electronic check
25. Payment Method_Mailed check


In [7]:
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test)
test_precision = precision_score(y_test, y_pred_test)
test_recall = recall_score(y_test, y_pred_test)
overfitting = train_accuracy - test_accuracy


print(f"=== LOGISTIC REGRESSION ===")
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test F1 Score:  {test_f1:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")
print(f"Overfitting:    {overfitting:.4f}")

=== LOGISTIC REGRESSION ===
Train Accuracy: 0.9102
Test Accuracy:  0.9106
Test F1 Score:  0.8346
Test Precision: 0.8196
Test Recall:    0.8503
Overfitting:    -0.0004


c:\Users\danie\OneDrive\Documentos\Daniel\FIAP Pós Tech\Fase 1\churn-prediction-mlp\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [8]:
model = DummyClassifier(random_state=42, strategy="most_frequent")
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test)
test_precision = precision_score(y_test, y_pred_test)
test_recall = recall_score(y_test, y_pred_test)
overfitting = train_accuracy - test_accuracy


print(f"=== Dummy Classifier ===")
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test F1 Score:  {test_f1:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")
print(f"Overfitting:    {overfitting:.4f}")


=== Dummy Classifier ===
Train Accuracy: 0.7346
Test Accuracy:  0.7346
Test F1 Score:  0.0000
Test Precision: 0.0000
Test Recall:    0.0000
Overfitting:    0.0001


c:\Users\danie\OneDrive\Documentos\Daniel\FIAP Pós Tech\Fase 1\churn-prediction-mlp\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Registra experimentos no MLFlow

In [9]:
import os
# Seta o caminho do banco de dados do MLflow para um arquivo SQLite localizado no diretório do projeto
mlflow_tracking_db = os.path.join(os.path.dirname(os.getcwd()), "mlflow.db")
mlflow_registry_uri = os.path.join(os.path.dirname(os.getcwd()), 'mlruns')

mlflow.set_tracking_uri(f"sqlite:///{mlflow_tracking_db}")
mlflow.set_registry_uri(f"file:///{mlflow_registry_uri}")

print(mlflow_tracking_db + "\n" + mlflow_registry_uri)

c:\Users\danie\OneDrive\Documentos\Daniel\FIAP Pós Tech\Fase 1\churn-prediction-mlp\mlflow.db
c:\Users\danie\OneDrive\Documentos\Daniel\FIAP Pós Tech\Fase 1\churn-prediction-mlp\mlruns


In [10]:
experiment_name = "chrun_prediction_logistic_regression"

experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    mlflow.create_experiment(
            name=experiment_name,
            artifact_location=f"file:///{mlflow_registry_uri}"
        )

    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name=experiment_name):
        model = LogisticRegression(random_state=42,  solver="liblinear")
        model.fit(X_train, y_train)

        y_pred_test = model.predict(X_test)

        test_accuracy = accuracy_score(y_test, y_pred_test)
        test_f1 = f1_score(y_test, y_pred_test)
        test_precision = precision_score(y_test, y_pred_test)
        test_recall = recall_score(y_test, y_pred_test)

        mlflow.log_param("model_type", "Logistic Regression")
        mlflow.log_param("selected_features", selected_features)
        mlflow.log_metric("train_accuracy", train_accuracy)
        mlflow.log_metric("test_accuracy", test_accuracy)
        mlflow.log_metric("test_f1_score", test_f1)
        mlflow.log_metric("test_precision", test_precision)
        mlflow.log_metric("test_recall", test_recall)
        mlflow.log_metric("overfitting", overfitting)

        dataset_path = '../data/processed/telco_customer_churn_processed.csv'
        mlflow.log_artifact(dataset_path, artifact_path="dataset")
        
        mlflow.sklearn.log_model(model, name="model")

        mlflow.set_tag("stage", "baseline")
        mlflow.set_tag("dataset", "telco_churn_processed")

2026/04/02 22:17:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [11]:
experiment_name = "chrun_prediction_dummy_classifier"

experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    mlflow.create_experiment(
            name=experiment_name,
            artifact_location=f"file:///{mlflow_registry_uri}"
        )

    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name=experiment_name):
        dummy = DummyClassifier(random_state=42, strategy="most_frequent")
        dummy.fit(X_train, y_train)

        y_pred_test = dummy.predict(X_test)

        test_accuracy = accuracy_score(y_test, y_pred_test)
        test_f1 = f1_score(y_test, y_pred_test)
        test_precision = precision_score(y_test, y_pred_test)
        test_recall = recall_score(y_test, y_pred_test)

        mlflow.log_param("model_type", "Dummy Classifier")
        mlflow.log_metric("train_accuracy", train_accuracy)
        mlflow.log_metric("test_accuracy", test_accuracy)
        mlflow.log_metric("test_f1_score", test_f1)
        mlflow.log_metric("test_precision", test_precision)
        mlflow.log_metric("test_recall", test_recall)
        mlflow.log_metric("overfitting", overfitting)

        dataset_path = '../data/processed/telco_customer_churn_processed.csv'
        mlflow.log_artifact(dataset_path, artifact_path="dataset")
        
        mlflow.sklearn.log_model(dummy, name="model")

        mlflow.set_tag("stage", "baseline")
        mlflow.set_tag("dataset", "telco_churn_processed")

c:\Users\danie\OneDrive\Documentos\Daniel\FIAP Pós Tech\Fase 1\churn-prediction-mlp\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
2026/04/02 22:17:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
